## 1. Imports

In [2]:
import pandas as pd
import math
import random
import time
import requests
import geopandas as gpd
from shapely.geometry import Point
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## 2. Chargement et filtrage Grand Est avec vrais contours

In [3]:
df = pd.read_csv('pharmacies_point.csv', low_memory=False, dtype={'addr-postcode': str})

# Conversion GPS EPSG:3857 -> latitude/longitude
coords = df['the_geom'].str.extract(r'POINT \(([^\s]+) ([^\s]+)\)')
df['x'] = coords[0].astype(float)
df['y'] = coords[1].astype(float)

def merc_to_latlon(x, y):
    lon = x / 20037508.34 * 180
    lat = math.degrees(2 * math.atan(math.exp(y / 20037508.34 * math.pi)) - math.pi / 2)
    return lat, lon

lats, lons = zip(*[merc_to_latlon(x, y) for x, y in zip(df['x'], df['y'])])
df['lat'] = lats
df['lon'] = lons

# Contours officiels des départements du Grand Est
url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
departements = gpd.read_file(url)
grand_est_depts = ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88']
grand_est = departements[departements['code'].isin(grand_est_depts)]
contour_grand_est = grand_est.geometry.union_all()

gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])],
    crs="EPSG:4326"
)

pharmacies_grand_est = gdf[gdf.geometry.within(contour_grand_est)].copy()
pharmacies_grand_est = pharmacies_grand_est[pharmacies_grand_est['name'].notna()].copy()
pharmacies_grand_est = pharmacies_grand_est.reset_index(drop=True)

print(f"Total pharmacies Grand Est : {len(pharmacies_grand_est)}")
print(f"Avec ville : {pharmacies_grand_est['addr-city'].notna().sum()}")
print(f"Sans ville : {pharmacies_grand_est['addr-city'].isna().sum()}")

Total pharmacies Grand Est : 1368
Avec ville : 248
Sans ville : 1120


## 3. Matching avec le fichier du prof

In [4]:
# Chargement du fichier du prof
df_prof = pd.read_excel('ListingPharmaciesGrandEstfeb2026.xlsx', header=1)
df_prof.columns = ['region', 'departement', 'code_insee', 'ville_prof', 'cp_prof', 'licence', 'adresse_prof']
df_prof['cp_prof'] = df_prof['cp_prof'].astype(str).str.zfill(5)
df_prof['ville_prof'] = df_prof['ville_prof'].str.strip().str.upper()

print(f"Pharmacies dans le fichier prof : {len(df_prof)}")

# Normalisation de notre fichier pour le matching
pharmacies_grand_est['addr-city-norm'] = pharmacies_grand_est['addr-city'].str.strip().str.upper()
pharmacies_grand_est['addr-postcode-norm'] = pharmacies_grand_est['addr-postcode'].astype(str).str.zfill(5)

# Matching par code postal + ville
matched = pharmacies_grand_est.merge(
    df_prof,
    left_on=['addr-postcode-norm', 'addr-city-norm'],
    right_on=['cp_prof', 'ville_prof'],
    how='outer',
    indicator=True
)

dans_les_deux   = matched[matched['_merge'] == 'both']
seulement_nous  = matched[matched['_merge'] == 'left_only']
seulement_prof  = matched[matched['_merge'] == 'right_only']

print(f"\n Dans les deux fichiers : {len(dans_les_deux)}")
print(f" Seulement dans notre fichier : {len(seulement_nous)}")
print(f" Seulement dans le fichier prof : {len(seulement_prof)}")

Pharmacies dans le fichier prof : 1558

 Dans les deux fichiers : 2014
 Seulement dans notre fichier : 1164
 Seulement dans le fichier prof : 1040


## 4. Géocodage des pharmacies manquantes

In [5]:
def geocoder_adresse(adresse, ville, cp):
    """Géocode une adresse via Nominatim pour obtenir lat/lon"""
    try:
        query = f"{adresse}, {cp} {ville}, France"
        url   = f"https://nominatim.openstreetmap.org/search?q={requests.utils.quote(query)}&format=json&limit=1"
        headers = {"User-Agent": "pharmacies-grand-est"}
        res  = requests.get(url, headers=headers, timeout=5)
        data = res.json()
        if data:
            return float(data[0]['lat']), float(data[0]['lon'])
        return None, None
    except:
        return None, None

# Géocoder les pharmacies du prof qui n'ont pas de coordonnées GPS
print("Géocodage des pharmacies manquantes...")
pharmacies_manquantes = seulement_prof.copy()
lats_new, lons_new = [], []

for i, (_, row) in enumerate(pharmacies_manquantes.iterrows()):
    lat, lon = geocoder_adresse(row['adresse_prof'], row['ville_prof'], row['cp_prof'])
    lats_new.append(lat)
    lons_new.append(lon)
    print(f"[{i+1}/{len(pharmacies_manquantes)}] {row['ville_prof']} → lat:{lat}, lon:{lon}", flush=True)
    time.sleep(0.8)  # respecter limite Nominatim

pharmacies_manquantes['lat'] = lats_new
pharmacies_manquantes['lon'] = lons_new
pharmacies_manquantes = pharmacies_manquantes[pharmacies_manquantes['lat'].notna()]

print(f"\n {len(pharmacies_manquantes)} pharmacies géocodées")

Géocodage des pharmacies manquantes...
[1/1040] CHARLEVILLE-MEZIERES → lat:49.7605244, lon:4.7183608
[2/1040] CHARLEVILLE-MEZIERES → lat:None, lon:None
[3/1040] CHARLEVILLE-MEZIERES → lat:49.7741185, lon:4.7212582
[4/1040] CHARLEVILLE-MEZIERES → lat:49.7651805, lon:4.7193036
[5/1040] CHARLEVILLE-MEZIERES → lat:49.7714744, lon:4.719894
[6/1040] CHARLEVILLE-MEZIERES → lat:49.7741921, lon:4.7184378
[7/1040] CHARLEVILLE-MEZIERES → lat:None, lon:None
[8/1040] CHARLEVILLE-MEZIERES → lat:49.775373, lon:4.713059
[9/1040] CHARLEVILLE-MEZIERES → lat:49.7504652, lon:4.7188967
[10/1040] CHARLEVILLE-MEZIERES → lat:49.7588418, lon:4.7228733
[11/1040] CHARLEVILLE-MEZIERES → lat:49.7469555, lon:4.7242123
[12/1040] CHARLEVILLE-MEZIERES → lat:49.7578007, lon:4.7187768
[13/1040] CHARLEVILLE-MEZIERES → lat:49.7808273, lon:4.7087951
[14/1040] CHARLEVILLE-MEZIERES → lat:49.7696724, lon:4.7244369
[15/1040] CHARLEVILLE-MEZIERES → lat:None, lon:None
[16/1040] CHARLEVILLE-MEZIERES → lat:49.7482544, lon:4.724246

## 5. Export fichier final Partie 1

In [6]:
# Recréer la colonne dept
pharmacies_grand_est['dept'] = pharmacies_grand_est['addr-postcode'].str[:2]

cols = ['name', 'addr-housenumber', 'addr-street', 'addr-city',
        'addr-postcode', 'dept', 'phone', 'website', 'opening_hours', 'lat', 'lon']
pharmacies_grand_est = pharmacies_grand_est[cols].reset_index(drop=True)

pharmacies_grand_est.to_csv('pharmacies_grand_est.csv', index=False)
print(f" Fichier exporté : {len(pharmacies_grand_est)} pharmacies")
print(pharmacies_grand_est.head())

 Fichier exporté : 1368 pharmacies
                      name addr-housenumber addr-street addr-city  \
0       Pharmacie Lorraine              NaN         NaN       NaN   
1        Pharmacie Jeannot              NaN         NaN       NaN   
2        Pharmacie Lambert              NaN         NaN       NaN   
3   Pharmacie des Villages              NaN         NaN       NaN   
4  Pharmacie Saint-Georges              NaN         NaN       NaN   

  addr-postcode dept phone website  \
0           NaN  NaN   NaN     NaN   
1           NaN  NaN   NaN     NaN   
2           NaN  NaN   NaN     NaN   
3           NaN  NaN   NaN     NaN   
4           NaN  NaN   NaN     NaN   

                                   opening_hours        lat       lon  
0                                            NaN  49.184730  6.895817  
1  Mo-Fr 08:00-12:30,13:30-19:15; Sa 08:30-17:00  49.118451  6.880416  
2                                            NaN  48.337782  7.324385  
3                                

## 6. Initialisation du driver Chrome

In [7]:
options = webdriver.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--lang=fr")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
print(" Driver Chrome initialisé !")

# Accepter cookies UNE FOIS
driver.get("https://www.google.com/maps?hl=fr")
time.sleep(5)

# Méthode 1
try:
    boutons = driver.find_elements(By.CSS_SELECTOR, "button")
    for b in boutons:
        if 'Tout accepter' in b.text:
            b.click()
            print("Cookies acceptés (méthode 1) !")
            time.sleep(3)
            break
except:
    pass

# Méthode 2 (iframe)
try:
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    for iframe in iframes:
        driver.switch_to.frame(iframe)
        for b in driver.find_elements(By.CSS_SELECTOR, "button"):
            if 'accepter' in b.text.lower() or 'accept' in b.text.lower():
                b.click()
                print(" Cookies acceptés (méthode 2) !")
                break
        driver.switch_to.default_content()
except:
    driver.switch_to.default_content()

# Vérification
time.sleep(3)
driver.get("https://www.google.com/maps/search/pharmacie/@48.5734,7.7521,17z?hl=fr")
time.sleep(5)
liens = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")
print(f" Vérification : {len(liens)} résultats trouvés")

 Driver Chrome initialisé !
Cookies acceptés (méthode 1) !
 Vérification : 7 résultats trouvés


## 7. Fonction scraping Google Maps

In [8]:
def get_avis_google(driver, nom, ville, cp, lat, lon):
    resultats = []
    note_moyenne = ""
    nb_avis = ""

    def score_resultat(label, nom, ville, cp):
        label = (label or "").lower()
        score = 0

        mots_nom = [m for m in nom.lower().replace("-", " ").split() if len(m) > 2]
        for mot in mots_nom:
            if mot in label:
                score += 3

        if ville and ville.lower() in label:
            score += 2

        if cp and cp in label:
            score += 2

        if "pharmacie" in label:
            score += 1

        return score

    try:
        query_parts = [nom, "pharmacie", "Grand Est", "France"]
        if ville and ville.lower() != "nan":
            query_parts.append(ville)
        if cp and cp.lower() != "nan":
            query_parts.append(cp)

        query = " ".join([str(x).strip() for x in query_parts if str(x).strip() != ""])

        url = f"https://www.google.com/maps/search/{requests.utils.quote(query)}/@{lat},{lon},17z?hl=fr"
        driver.get(url)
        time.sleep(random.uniform(2, 3))

        try:
            for b in driver.find_elements(By.CSS_SELECTOR, "button"):
                if "Tout accepter" in b.text:
                    b.click()
                    time.sleep(2)
                    break
        except:
            pass

        liens = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")

        if not liens:
            url2 = f"https://www.google.com/maps/search/pharmacie/@{lat},{lon},16z?hl=fr"
            driver.get(url2)
            time.sleep(random.uniform(2, 3))
            liens = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")

        if not liens:
            print("  Aucun résultat", flush=True)
            return {"note_moyenne": "", "nb_avis": "", "avis": []}

        best = None
        best_score = -1

        for lien in liens:
            label = lien.get_attribute("aria-label") or ""
            s = score_resultat(label, nom, ville, cp)
            if s > best_score:
                best_score = s
                best = lien

        if best is None or best_score < 3:
            print("  Résultat trop ambigu, on ignore cette pharmacie", flush=True)
            return {"note_moyenne": "", "nb_avis": "", "avis": []}

        print(f"  Clic sur : {best.get_attribute('aria-label')}", flush=True)

        driver.execute_script("arguments[0].click();", best)
        time.sleep(random.uniform(3, 4))

        tabs = []
        for attempt in range(15):
            tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
            if len(tabs) >= 2:
                break
            time.sleep(1)

        try:
            note_moyenne = driver.find_element(By.CSS_SELECTOR, "div.F7nice span").text
        except:
            pass

        try:
            nb_avis = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-label]").get_attribute("aria-label")
        except:
            pass

        onglet_trouve = False
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")

        for t in tabs:
            if "avis" in t.text.lower():
                t.click()
                onglet_trouve = True
                time.sleep(random.uniform(1.5, 2.5))
                break

        if not onglet_trouve:
            for t in tabs:
                label = (t.get_attribute("aria-label") or "").lower()
                if "avis" in label or "review" in label:
                    t.click()
                    onglet_trouve = True
                    time.sleep(random.uniform(1.5, 2.5))
                    break

        if not onglet_trouve:
            print(f"  Onglet Avis non trouvé ({len(tabs)} onglets)", flush=True)
            return {"note_moyenne": note_moyenne, "nb_avis": nb_avis, "avis": resultats}

        scrollable = None
        for selector in ["div.m6QErb.DxyBCb", "div.m6QErb", "div[role='feed']"]:
            try:
                scrollable = driver.find_element(By.CSS_SELECTOR, selector)
                break
            except:
                continue

        for i in range(4):
            if scrollable:
                driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scrollable)
            time.sleep(random.uniform(0.5, 1))
            for btn in driver.find_elements(By.CSS_SELECTOR, "button.w8nwRe.kyuRq"):
                try:
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(0.2)
                except:
                    pass

        avis_elements = []
        for selector in [
            "div.jftiEf",
            "div[data-review-id]",
            "div.GHT2ce",
            "div.jJc9Ad",
            "div.gws-localreviews__google-review"
        ]:
            avis_elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if avis_elements:
                break

        if not avis_elements:
            textes = driver.find_elements(By.CSS_SELECTOR, "span.wiI7pd")
            auteurs = driver.find_elements(By.CSS_SELECTOR, "div.d4r55")
            notes = driver.find_elements(By.CSS_SELECTOR, "span.kvMYJc")
            dates = driver.find_elements(By.CSS_SELECTOR, "span.rsqaWe")
            nb = max(len(textes), len(auteurs))

            if nb > 0:
                print(f"  {nb} avis (méthode directe)", flush=True)

            for j in range(nb):
                resultats.append({
                    "auteur": auteurs[j].text if j < len(auteurs) else "",
                    "note_avis": notes[j].get_attribute("aria-label") if j < len(notes) else "",
                    "date": dates[j].text if j < len(dates) else "",
                    "avis_texte": textes[j].text if j < len(textes) else "",
                    "reponse_owner": ""
                })

            return {"note_moyenne": note_moyenne, "nb_avis": nb_avis, "avis": resultats}

        avis_elements = avis_elements[:3]

        print(f"  {len(avis_elements)} avis trouvés", flush=True)

        for avis in avis_elements:
            try:
                plus_btn = avis.find_element(By.CSS_SELECTOR, "button.w8nwRe")
                driver.execute_script("arguments[0].click();", plus_btn)
                time.sleep(0.3)
            except:
                pass

            auteur, note, date, texte, reponse = "", "", "", "", ""

            try:
                auteur = avis.find_element(By.CSS_SELECTOR, "div.d4r55").text
            except:
                pass

            try:
                note = avis.find_element(By.CSS_SELECTOR, "span.kvMYJc").get_attribute("aria-label")
            except:
                pass

            try:
                date = avis.find_element(By.CSS_SELECTOR, "span.rsqaWe").text
            except:
                pass

            try:
                texte = avis.find_element(By.CSS_SELECTOR, "span.wiI7pd").text
            except:
                pass

            try:
                reponse = avis.find_element(By.CSS_SELECTOR, "div.CDe7pd").text
            except:
                pass

            resultats.append({
                "auteur": auteur,
                "note_avis": note,
                "date": date,
                "avis_texte": texte,
                "reponse_owner": reponse
            })

        return {"note_moyenne": note_moyenne, "nb_avis": nb_avis, "avis": resultats}

    except Exception as e:
        print(f"   Erreur : {e}", flush=True)
        return {"note_moyenne": "", "nb_avis": "", "avis": []}

## 8. Boucle par lot + export

In [36]:
def creer_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36")
    d = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    d.get("https://www.google.com/maps?hl=fr")
    time.sleep(5)

    try:
        for b in d.find_elements(By.CSS_SELECTOR, "button"):
            if "Tout accepter" in b.text:
                b.click()
                time.sleep(3)
                break
    except:
        pass

    return d


pharmacies_grand_est['name'] = pharmacies_grand_est['name'].fillna('').astype(str).str.strip()
pharmacies_grand_est['addr-city'] = pharmacies_grand_est['addr-city'].fillna('').astype(str).str.strip()
pharmacies_grand_est['addr-postcode'] = pharmacies_grand_est['addr-postcode'].fillna('').astype(str).str.strip()

pharmacies_grand_est = pharmacies_grand_est[
    (pharmacies_grand_est['name'] != '')
].copy()

BATCH_SIZE = 100
START = 1300

pharmacies_batch = pharmacies_grand_est.iloc[START:START + BATCH_SIZE]
all_avis = []
driver = creer_driver()

for i, (_, row) in enumerate(pharmacies_batch.iterrows()):
    nom = str(row.get('name', '') or '').strip()
    ville = str(row.get('addr-city', '') or '').strip()
    cp = str(row.get('addr-postcode', '') or '').strip()
    lat = row.get('lat', None)
    lon = row.get('lon', None)

    if ville.lower() == 'nan':
        ville = ''
    if cp.lower() == 'nan':
        cp = ''

    print(f"\n[{i+1}/{len(pharmacies_batch)}] {nom} - {ville} ({cp})", flush=True)

    try:
        driver.current_url
    except:
        print(" Chrome a planté, relance...", flush=True)
        try:
            driver.quit()
        except:
            pass
        driver = creer_driver()
        print(" Chrome relancé !", flush=True)

    if i > 0 and i % 20 == 0:
        print(" Restart préventif...", flush=True)
        try:
            driver.quit()
        except:
            pass
        driver = creer_driver()
        print(" Chrome relancé !", flush=True)

    data = get_avis_google(driver, nom, ville, cp, lat, lon)

    note_moyenne = data.get("note_moyenne", "")
    nb_avis_txt = data.get("nb_avis", "")
    avis_list = data.get("avis", [])

    print(f" → {len(avis_list)} avis récupérés", flush=True)

    if avis_list:
        for a in avis_list:
            if str(a.get("avis_texte", "")).strip() != "":
                all_avis.append({
                    "pharmacie": nom,
                    "ville": ville,
                    "code_postal": cp,
                    "latitude": lat,
                    "longitude": lon,
                    "note_moyenne": note_moyenne,
                    "nombre_avis": nb_avis_txt,
                    "avis_texte": a.get("avis_texte", ""),
                    "reponse_owner": a.get("reponse_owner", ""),
                    "auteur": a.get("auteur", ""),
                    "note_avis": a.get("note_avis", ""),
                    "date": a.get("date", "")
                })

    if (i + 1) % 10 == 0:
        df_temp = pd.DataFrame(all_avis)
        if not df_temp.empty:
            df_temp = df_temp.drop_duplicates(
                subset=["pharmacie", "ville", "avis_texte", "auteur", "date"]
            )
        df_temp.to_csv(f"avis_google_temp_{i+1}.csv", index=False, encoding="utf-8-sig")
        print(f" Sauvegarde intermédiaire : {len(df_temp)} lignes", flush=True)

    time.sleep(random.uniform(1, 2))

print(f"\n Lot {START}-{START+BATCH_SIZE} terminé !")

df_avis = pd.DataFrame(all_avis)

if not df_avis.empty:
    df_avis = df_avis.drop_duplicates(
        subset=["pharmacie", "ville", "avis_texte", "auteur", "date"]
    )
    df_avis = df_avis[df_avis["avis_texte"].fillna("").str.strip() != ""]
    df_avis = df_avis.sort_values(["pharmacie", "date"], ascending=[True, True])

df_avis.to_excel(f"avis_google_{START}_{START+BATCH_SIZE}_propre.xlsx", index=False)
df_avis.to_csv(f"avis_google_{START}_{START+BATCH_SIZE}_propre.csv", index=False, encoding="utf-8-sig")
try:
    driver.quit()
except:
    pass
print(df_avis.head(10))
print(f"Nombre de lignes finales : {len(df_avis)}")


[1/68] Pharmacie Baclet Dubois - Châlons-en-Champagne (51000)
  Clic sur : Pharmacie du Mont Saint Michel
  3 avis trouvés
 → 3 avis récupérés

[2/68] Pharmacie du Cerf -  ()
  Clic sur : Pharmacie du Cerf
  3 avis trouvés
 → 3 avis récupérés

[3/68] Pharmacie de la Marne -  ()
  Clic sur : Pharmacie De La Marne
  3 avis trouvés
 → 3 avis récupérés

[4/68] Pharmacie Chardard -  ()
  Clic sur : Pharmacie du Lys
  3 avis trouvés
 → 3 avis récupérés

[5/68] Pharmacie du Cavalier -  ()
  Clic sur : Pharmacie Du Cavalier
  3 avis trouvés
 → 3 avis récupérés

[6/68] Pharmacie Saint-Jean -  ()
  Clic sur : Pharmacie SAINT JEAN
 → 0 avis récupérés

[7/68] Pharmacie Wilhelm -  ()
  Clic sur : Pharmacie Du 80 Route De L'Hôpital Anton&Willem - Herboristerie
 → 0 avis récupérés

[8/68] Pharmacie de la Mauchère -  ()
  Clic sur : Pharmacie de la Mauchère
  3 avis trouvés
 → 3 avis récupérés

[9/68] Pharmacie Le Danvic -  ()
  Clic sur : Pharmacie le Danvic
  3 avis trouvés
 → 3 avis récupérés

[10

In [35]:
df = pd.read_csv("avis_google_0_100_propre.csv")

df["pharmacie"] = df["pharmacie"].fillna("").astype(str).str.strip()
df["ville"] = df["ville"].fillna("").astype(str).str.strip()
df["code_postal"] = df["code_postal"].fillna("").astype(str).str.replace(".0", "", regex=False).str.strip()
df["avis_texte"] = df["avis_texte"].fillna("").astype(str).str.strip()
df["auteur"] = df["auteur"].fillna("").astype(str).str.strip()
df["date"] = df["date"].fillna("").astype(str).str.strip()

df_fiable = df[
    (df["ville"] != "") | (df["code_postal"] != "")
].copy()

df_fiable = df_fiable.drop_duplicates(
    subset=["pharmacie", "ville", "code_postal", "avis_texte", "auteur", "date"]
).copy()

noms_ambigus = [
    "Pharmacie de l'Europe",
    "Pharmacie de la Place",
    "Pharmacie Mutualiste",
    "Pharmacie Saint-Georges"
]

df_fiable = df_fiable[~df_fiable["pharmacie"].isin(noms_ambigus)].copy()

df_fiable = df_fiable.sort_values(
    by=["pharmacie", "ville", "date"],
    ascending=[True, True, True]
)

df_fiable.to_csv("avis_google_0_100_presentation.csv", index=False, encoding="utf-8-sig")
df_fiable.to_excel("avis_google_0_100_presentation.xlsx", index=False)

print("Lot nettoyé :", len(df_fiable))
print("Pharmacies distinctes :", df_fiable["pharmacie"].nunique())

Lot nettoyé : 47
Pharmacies distinctes : 16


In [2]:
import pandas as pd
import glob
import re

# Récupère tous les fichiers
fichiers_csv = glob.glob("avis_google_*_propre.csv")

# Tri numérique correct par le premier nombre
def extraire_debut(nom):
    match = re.search(r'avis_google_(\d+)_(\d+)', nom)
    return int(match.group(1)) if match else 0

fichiers_csv.sort(key=extraire_debut)

# Garder uniquement les fichiers sans chevauchement
# On garde le fichier avec le plus grand écart pour chaque plage
fichiers_filtres = []
dernier_fin = -1

for fichier in fichiers_csv:
    match = re.search(r'avis_google_(\d+)_(\d+)', fichier)
    debut = int(match.group(1))
    fin   = int(match.group(2))
    
    if debut >= dernier_fin:  # pas de chevauchement
        fichiers_filtres.append(fichier)
        dernier_fin = fin
        print(f" Gardé  : {fichier} ({debut}-{fin})")
    else:
        print(f" Ignoré : {fichier} (doublon avec plage précédente)")

# Fusion
dfs = []
for fichier in fichiers_filtres:
    df_temp = pd.read_csv(fichier)
    print(f"{fichier} → {len(df_temp)} lignes")
    dfs.append(df_temp)

df_final = pd.concat(dfs, ignore_index=True)

# Supprimer les doublons éventuels
df_final = df_final.drop_duplicates()

print(f"\n Total lignes fusionnées : {len(df_final)}")
print(f" Total pharmacies uniques : {df_final['pharmacie'].nunique()}")

# Export
df_final.to_csv("pharmacies_avis_FINAL.csv", index=False)
df_final.to_excel("pharmacies_avis_FINAL.xlsx", index=False)
print(" Fichier CSV créé : pharmacies_avis_FINAL.csv")
print(" Fichier Excel créé : pharmacies_avis_FINAL.xlsx")

 Gardé  : avis_google_0_100_propre.csv (0-100)
 Ignoré : avis_google_0_20_propre.csv (doublon avec plage précédente)
 Ignoré : avis_google_20_30_propre.csv (doublon avec plage précédente)
 Gardé  : avis_google_100_200_propre.csv (100-200)
 Gardé  : avis_google_200_300_propre.csv (200-300)
 Gardé  : avis_google_300_400_propre.csv (300-400)
 Gardé  : avis_google_400_500_propre.csv (400-500)
 Gardé  : avis_google_500_600_propre.csv (500-600)
 Gardé  : avis_google_600_700_propre.csv (600-700)
 Gardé  : avis_google_700_800_propre.csv (700-800)
 Gardé  : avis_google_800_900_propre.csv (800-900)
 Gardé  : avis_google_900_1000_propre.csv (900-1000)
 Gardé  : avis_google_1000_1100_propre.csv (1000-1100)
 Gardé  : avis_google_1100_1200_propre.csv (1100-1200)
 Gardé  : avis_google_1200_1300_propre.csv (1200-1300)
 Gardé  : avis_google_1300_1400_propre.csv (1300-1400)
avis_google_0_100_propre.csv → 218 lignes
avis_google_100_200_propre.csv → 198 lignes
avis_google_200_300_propre.csv → 221 lignes
a